In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier



In [2]:
train_path = "/kaggle/input/tabular-playground-series-dec-2021/train.csv"
test_path = "/kaggle/input/tabular-playground-series-dec-2021/test.csv"
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)



In [3]:
# Encode target labels and compute the correct number of classes
X = train_df.drop(columns=["Cover_Type", "Id"])
X_test = test_df.drop(columns=["Id"])

y_enc, class_labels = pd.factorize(train_df["Cover_Type"])
y_enc = y_enc.astype(int)

# Ensure num_classes matches the actual number of distinct labels
num_classes = len(class_labels)

unique, counts = np.unique(y_enc, return_counts=True)
if np.min(counts) >= 2:
    x_train, x_val, y_train, y_val = train_test_split(
        X, y_enc, test_size=0.2, random_state=42, shuffle=True, stratify=y_enc
    )
else:
    x_train, x_val, y_train, y_val = train_test_split(
        X, y_enc, test_size=0.2, random_state=42, shuffle=True, stratify=None
    )



In [4]:
# Slightly increase the number of trees to boost accuracy while keeping the original architecture
model_xgbc = XGBClassifier(
    n_estimators=300,  # increased from 200
    learning_rate=0.1,
    max_depth=8,
    n_jobs=4,
    tree_method="hist",
    objective="multi:softprob",
    eval_metric="merror",
    num_class=num_classes,
    use_label_encoder=False,
)



In [5]:
model_xgbc.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)



XGBoostError: [14:50:09] /workspace/src/metric/multiclass_metric.cu:35: Check failed: label_error >= 0 && label_error < static_cast<int32_t>(n_class): MultiClassEvaluation: label must be in [0, num_class), num_class=6 but found 6 in label
Stack trace:
  [bt] (0) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x1ba24e) [0x7fff836ce24e]
  [bt] (1) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x9a66f6) [0x7fff83eba6f6]
  [bt] (2) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x4efea5) [0x7fff83a03ea5]
  [bt] (3) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x4f0168) [0x7fff83a04168]
  [bt] (4) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x4c8ab0) [0x7fff839dcab0]
  [bt] (5) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(XGBoosterEvalOneIter+0x2f6) [0x7fff8367ebb6]
  [bt] (6) /lib/x86_64-linux-gnu/libffi.so.8(+0x7e2e) [0x7ffff63ace2e]
  [bt] (7) /lib/x86_64-linux-g

In [6]:
val_pred_enc = model_xgbc.predict(x_val)
val_pred = class_labels[val_pred_enc]  # map back to original labels
val_true = class_labels[y_val]
val_acc = accuracy_score(val_true, val_pred)
print(f"Validation accuracy: {val_acc:.5f}")



NotFittedError: need to call fit or load_model beforehand

In [7]:
test_pred_enc = model_xgbc.predict(X_test)
test_pred = class_labels[test_pred_enc].astype(int)



NotFittedError: need to call fit or load_model beforehand

In [8]:
result = pd.DataFrame({"Id": test_df["Id"], "Cover_Type": test_pred})



NameError: name 'test_pred' is not defined

In [9]:
output_path = "/kaggle/working/submission.csv"
result.to_csv(output_path, index=False)
print(f"Submission saved to {output_path}")

NameError: name 'result' is not defined